<a href="https://colab.research.google.com/github/Maryum58/AI-Code-Reviewer/blob/main/py_code_reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Code Reviewer

This project analyzes Python code using AST (Abstract Syntax Tree)
and provides automated feedback on code quality, structure, and best practices.

### Features:
- Static code analysis
- Code quality detection
- Automated suggestions
- Code scoring system

In [ ]:
!pip install transformers torch

In [ ]:
import ast

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Salesforce/codegen-350M-mono"
)

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

In [8]:
def analyze_code(code):
    issues = []

    try:
        tree = ast.parse(code)
    except:
        return ["Invalid Python code"]

    for node in ast.walk(tree):

        # Long functions
        if isinstance(node, ast.FunctionDef):
            if len(node.body) > 8:
                issues.append(f"Function '{node.name}' is too long (>8 lines)")

        # Too many arguments
        if isinstance(node, ast.FunctionDef):
            if len(node.args.args) > 4:
                issues.append(f"Function '{node.name}' has too many parameters")

        # Nested ifs
        if isinstance(node, ast.If):
            if len(node.body) > 2:
                issues.append("Deep nesting detected")

        # Bad variable names
        if isinstance(node, ast.Name):
            if len(node.id) == 1:
                issues.append(f"Variable '{node.id}' name is not descriptive")

    return list(set(issues))

In [9]:
def ai_explain(code, issues):
    prompt = f"""
# Python Code Review

## Code:
{code}

## Issues:
{issues}

## Review:
- Explain what the code does
- Explain the issues clearly
- Suggest improvements

Answer in clear bullet points:
"""

    result = generator(
        prompt,
        max_length=200,
        truncation=True,
        temperature=0.3
    )

    return result[0]['generated_text']

In [10]:
def summarize_code(code):
    num_functions = code.count("def ")
    num_lines = len(code.splitlines())
    return f"This code has {num_functions} function(s) and {num_lines} line(s)."

print("\n📄 Code Summary:")
print(summarize_code(code))


📄 Code Summary:
This code has 1 function(s) and 9 line(s).


In [11]:
def suggest_fixes(issues):
    suggestions = []

    for issue in issues:
        if "too long" in issue:
            suggestions.append("Consider modularizing the function into smaller reusable components")

        elif "too many parameters" in issue:
            suggestions.append("Refactor using objects or grouped parameters to improve readability")

        elif "nesting" in issue:
            suggestions.append("Reduce nesting by applying early returns or restructuring logic")

        elif "not descriptive" in issue:
            suggestions.append("Use meaningful variable names to improve maintainability")

    return list(set(suggestions))

In [12]:
def clean_output(text):
    return text.split("## Review:")[-1].strip()

In [13]:
# Paste test code here
code = """
def example():
    x = 10
    y = 20
    z = 30
    if x > 5:
        if y > 10:
            if z > 15:
                print("Deep nesting")
"""

issues = analyze_code(code)
suggestions = suggest_fixes(issues)

print("🔍 Issues Found:")
for i in issues:
    print("-", i)

print("\n💡 Suggestions:")
for s in suggestions:
    print("-", s)

score = max(0, 10 - len(issues))
print(f"\n⭐ Code Score: {score}/10")

🔍 Issues Found:
- Variable 'x' name is not descriptive
- Variable 'z' name is not descriptive
- Variable 'y' name is not descriptive

💡 Suggestions:
- Use meaningful variable names to improve maintainability

⭐ Code Score: 7/10
